# FAISS

In [155]:
documents = [
    "Python is a programming language.",
    "Python is commonly used for machine learning and data science.",
    "Vector databases store numerical representations of text called embeddings.",
    "Embeddings represent the meaning of text as numbers.",
    "Chroma and FAISS can be used for similarity search."
]

Create embeddings

In [156]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)


### `faiss.IndexFlatL2`

Lower = closer

In [168]:
import faiss
import numpy as np

vectors = np.array(embeddings).astype("float32")

Create L2 FAISS index

In [169]:
index = faiss.IndexFlatL2(vectors.shape[1])

Store embeddings in FAISS

In [170]:
index.add(vectors)
print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 5


Save FAISS

In [171]:
faiss.write_index(index, "my_vectors.faiss")
print("FAISS index saved")

FAISS index saved


Load FAISS

In [172]:
index = faiss.read_index("my_vectors.faiss")
print("Vectors:", index.ntotal)

Vectors: 5


User query

In [173]:
query = "What is Python used for?"
query_vector = model.encode([query]).astype("float32")

Search FAISS

In [174]:
distances, indexes = index.search(
    query_vector, 
    k=3
)

for i, distance in zip(indexes[0], distances[0]):
    print(f"Distance: {distance:.4f}")
    print(f"Text: {documents[i]}")
    print("-" * 50)

Distance: 0.3126
Text: Python is a programming language.
--------------------------------------------------
Distance: 0.3978
Text: Python is commonly used for machine learning and data science.
--------------------------------------------------
Distance: 1.6162
Text: Vector databases store numerical representations of text called embeddings.
--------------------------------------------------


In [175]:
distances, indices = index.search(
    query_vector,
    k=3
)

for rank, (idx, distance) in enumerate(
    zip(indices[0], distances[0]),
    start=1
):
    print(f"Rank: {rank}")
    print(f"Distance: {distance:.4f}")
    print(f"Document: {documents[idx]}")
    print("-" * 50)

Rank: 1
Distance: 0.3126
Document: Python is a programming language.
--------------------------------------------------
Rank: 2
Distance: 0.3978
Document: Python is commonly used for machine learning and data science.
--------------------------------------------------
Rank: 3
Distance: 1.6162
Document: Vector databases store numerical representations of text called embeddings.
--------------------------------------------------


In [176]:
print(index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001A416189CF0> >


### `faiss.IndexFlatL2()`

Higher = more similar

In [177]:
import faiss
import numpy as np

vectors = np.array(embeddings).astype("float32")

Normalize document vectors

In [178]:
faiss.normalize_L2(vectors)

Create Inner Product index

In [179]:
index = faiss.IndexFlatIP(vectors.shape[1])

Store embeddings in FAISS

In [180]:
index.add(vectors)
print("Vectors:", index.ntotal)

Vectors: 5


User query

In [181]:
query = "What is Python used for?"
query_embedding = model.encode([query]).astype("float32")

Normalize query vector

In [182]:
faiss.normalize_L2(query_vector)

Search

In [183]:
distances, indices = index.search(
    query_embedding,
    k=3
)

for rank, (idx, distances) in enumerate(
    zip(indices[0], distances[0]),
    start=1
):
    print(f"Rank: {rank}")
    print(f"Distances: {distances:.4f}")
    print(f"Document: {documents[idx]}")
    print("-" * 50)

Rank: 1
Distances: 0.8437
Document: Python is a programming language.
--------------------------------------------------
Rank: 2
Distances: 0.8011
Document: Python is commonly used for machine learning and data science.
--------------------------------------------------
Rank: 3
Distances: 0.1919
Document: Vector databases store numerical representations of text called embeddings.
--------------------------------------------------


In [184]:
print(index)

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001A416192470> >


### `faiss.IndexHNSWFlat`

In [185]:
import faiss
import numpy as np

vectors = embeddings.astype("float32")

faiss.normalize_L2(vectors)

Create HNSW Index

In [186]:
dimension = vectors.shape[1]

index = faiss.IndexHNSWFlat(
    dimension,
    32
)

Add vectors

In [187]:
index.add(vectors)
print(index.ntotal)

5


In [188]:
query = "What is Python used for?"

query_vector = model.encode([query]).astype("float32")

faiss.normalize_L2(query_vector)

Search

In [189]:
scores, indices = index.search(
    query_vector,
    k=3
)

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    print(f"Rank: {rank}")
    print(f"Score: {score:.4f}")
    print(f"Document: {documents[idx]}")
    print("-" * 50)

Rank: 1
Score: 0.3126
Document: Python is a programming language.
--------------------------------------------------
Rank: 2
Score: 0.3978
Document: Python is commonly used for machine learning and data science.
--------------------------------------------------
Rank: 3
Score: 1.6162
Document: Vector databases store numerical representations of text called embeddings.
--------------------------------------------------


In [190]:
print(index)

<faiss.swigfaiss.IndexHNSWFlat; proxy of <Swig Object of type 'faiss::IndexHNSWFlat *' at 0x000001A419EAB0B0> >
